

# <div style="text-align:center; border-radius:15px; padding:15px; margin:0; font-size:100%; font-family: 'Arial Black', Arial, sans-serif; text-shadow: 2px 2px 5px rgba(0,0,0,0.7); background: linear-gradient(90deg, #0f2027, #203a43, #2c5364); overflow:hidden; box-shadow:0 2px 5px rgba(0, 0, 0, 0.3); color:white;"><b>2.1 Data Load
</b></div>

In [ ]:
import pandas as pd
import numpy as np

import joblib
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from joblib import dump
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold


import optuna.visualization as vis



In [26]:
df = pd.read_csv('../Dataset/credit_risk_data_cleaned.csv')
print("dataset shape:", df.shape)

dataset shape: (91046, 12)




# <div style="text-align:center; border-radius:15px; padding:15px; margin:0; font-size:100%; font-family: 'Arial Black', Arial, sans-serif; text-shadow: 2px 2px 5px rgba(0,0,0,0.7); background: linear-gradient(90deg, #0f2027, #203a43, #2c5364); overflow:hidden; box-shadow:0 2px 5px rgba(0, 0, 0, 0.3); color:white;"><b>2.2 Feature Engineering and split
</b></div>

In [27]:
def feature_engineering(df):
    df['income_to_loan'] = df['person_income'] / (df['loan_amnt'] + 1)
    df['age_to_emp_length'] = df['person_age'] / (df['person_emp_length'] + 1)
    df['log_income'] = np.log1p(df['person_income'])
    df['age_to_credit_history'] = df['person_age'] / (df['cb_person_cred_hist_length'] )
    df['stability_score'] = (df['person_emp_length'] * df['person_income']) / (df['loan_amnt'] * (df['cb_person_cred_hist_length'] + 1))
    df['rate_to_age'] = df['loan_int_rate'] / df['person_age']
    return df
df = feature_engineering(df)
print("After feature engineering:", df.shape)

After feature engineering: (91046, 18)


In [28]:

cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical Columns:", cat_cols)
print("Numerical Columns:", num_cols)
print()

X = df.drop(columns=['loan_status'])
y = df['loan_status']
print("Features shape:", X.shape)
print("Target shape:", y.shape)
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print("Features after encoding shape:", X.shape)

Categorical Columns: ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
Numerical Columns: ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'loan_status', 'income_to_loan', 'age_to_emp_length', 'log_income', 'age_to_credit_history', 'stability_score', 'rate_to_age']

Features shape: (91046, 17)
Target shape: (91046,)
Features after encoding shape: (91046, 28)


In [29]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)

Training set shape: (72836, 28) (72836,)
Test set shape: (18210, 28) (18210,)


In [30]:
joblib.dump(X.columns, '../feature_columns.pkl')
print("Feature columns saved to feature_columns.pkl")

joblib.dump([X_train, X_test, y_train, y_test], '../train_test_split.pkl')

print("Successfully dumped all 4 sets to a single file.")

Feature columns saved to feature_columns.pkl
Successfully dumped all 4 sets to a single file.




# <div style="text-align:center; border-radius:15px; padding:15px; margin:0; font-size:100%; font-family: 'Arial Black', Arial, sans-serif; text-shadow: 2px 2px 5px rgba(0,0,0,0.7); background: linear-gradient(90deg, #0f2027, #203a43, #2c5364); overflow:hidden; box-shadow:0 2px 5px rgba(0, 0, 0, 0.3); color:white;"><b>2.3 Models
</b></div>

In [31]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Logistic regression

In [32]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])
lr_pipeline.fit(X_train, y_train)
lr_score = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring='roc_auc')
print("Logistic Regression CV AUC Scores:", lr_score)
print(f"Mean AUC: {np.mean(lr_score):.4f}")
print(f"Std AUC: {np.std(lr_score):.4f}")



Logistic Regression CV AUC Scores: [0.89518224 0.88743268 0.88885467 0.8922069  0.90124385]
Mean AUC: 0.8930
Std AUC: 0.0049


## Random Forest

In [33]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_scores = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='roc_auc')
print("Random Forest CV AUC Scores:", rf_scores)
print(f"Mean AUC : {np.mean(rf_scores):.4f}")
print(f"Std AUC : {np.std(rf_scores):.4f}")

Random Forest CV AUC Scores: [0.9372418  0.94072426 0.93615487 0.93785543 0.93867795]
Mean AUC : 0.9381
Std AUC : 0.0015


## XGB

In [34]:
xgb_model = XGBClassifier(random_state=42)
xgb_scores = cross_val_score(xgb_model, X_train, y_train, cv=kf, scoring='roc_auc')
print("XGBoost CV AUC Scores:", xgb_scores)
print(f"Mean AUC : {np.mean(xgb_scores):.4f}")
print(f"Std AUC : {np.std(xgb_scores):.4f}")

XGBoost CV AUC Scores: [0.9536019  0.95512485 0.95330005 0.95455483 0.95238631]
Mean AUC : 0.9538
Std AUC : 0.0010


## Baseline Comparison

| Model | CV Mean AUC |
|---|---|
| Logistic Regression | 0.893 |
| Random Forest | 0.938 |
| XGBoost  | 0.954 ✅|

Starting with Logistic Regression gave us a solid linear baseline at 0.893. 
Random Forest pushed that further to 0.938 by capturing non-linear patterns 
in the data. XGBoost took it a step further, reaching 0.954 making it 
the natural choice to carry forward into hyperparameter tuning.



# <div style="text-align:center; border-radius:15px; padding:15px; margin:0; font-size:100%; font-family: 'Arial Black', Arial, sans-serif; text-shadow: 2px 2px 5px rgba(0,0,0,0.7); background: linear-gradient(90deg, #0f2027, #203a43, #2c5364); overflow:hidden; box-shadow:0 2px 5px rgba(0, 0, 0, 0.3); color:white;"><b>2.4 Hyperparameter Tuning (Optuna)
</b></div>

In [ ]:
def xgb_objective(trial):
    params = {
        # Number of boosting rounds
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 1000),
        # Maximum depth of each tree
        'max_depth'       : trial.suggest_int('max_depth', 3, 15),
        # Step size shrinkage
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        # Fraction of training samples per tree
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        # Fraction of features per tree
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        # Minimum instance weight in a leaf
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        # L2 regularization
        'reg_lambda'      : trial.suggest_float('reg_lambda', 0.0, 5.0),
        # L1 regularization
        'reg_alpha'       : trial.suggest_float('reg_alpha', 0.0, 5.0),
        # Class imbalance weight
        'scale_pos_weight': y_train.value_counts()[0] / y_train.value_counts()[1],
        'random_state'    : 42,
        'eval_metric'     : 'logloss',
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='roc_auc', n_jobs=-1)
    return np.mean(scores)

sampler = optuna.samplers.TPESampler(seed=42)
study   = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(xgb_objective, n_trials=150)

print(f'Best CV AUC    : {study.best_value:.4f}')
print(f'Best XGB Params: {study.best_params}')


In [ ]:
vis.plot_optimization_history(study).show()

vis.plot_param_importances(study).show()